In [123]:
import torch
import matplotlib.pyplot as plt
import numpy as np

In [124]:
X = torch.tensor([
    [1.0, 2.0],
    [2.0, 4.0],
    [3.0, 6.0],
    [4.0, 8.0],
    [5.0, 10.0]
])

In [125]:
mean = X.mean(dim=0)
mean

tensor([3., 6.])

In [126]:
X_c = X-mean
X_c

tensor([[-2., -4.],
        [-1., -2.],
        [ 0.,  0.],
        [ 1.,  2.],
        [ 2.,  4.]])

In [127]:
# we need covariance matrix first
# we can do torch.cov() but to learn we shall implement the actual code

$$\mathbf{\Sigma} = \frac{1}{N} \mathbf{X}_c^T \mathbf{X}_c$$

In [128]:
# Original X
#    ↓
# Center X
#    ↓
# Covariance matrix
#    ↓
# Eigenvectors + Eigenvalues
#    ↓
# Sort by eigenvalues
#    ↓
# Choose top components
#    ↓
# Project X onto them
#    ↓
# Reduced X

# this is the procedure basically for now and you know where we stand

In [129]:
cov_matrix = (X_c.T @ X_c)/len(X)

In [130]:
cov_matrix

tensor([[2., 4.],
        [4., 8.]])

In [131]:
# now coming to eigen bhai log
eigenval, eigenvec = torch.linalg.eigh(cov_matrix)

In [132]:
eigenval, eigenvec

(tensor([ 0., 10.]),
 tensor([[-0.8944,  0.4472],
         [ 0.4472,  0.8944]]))

In [133]:
# bcoz eigh() sorts asc, we need to reverse the order before using the components
eigenval = eigenval.flip(0)
eigenvec = eigenvec.flip(1)

print(eigenval)
print(eigenvec)

tensor([10.,  0.])
tensor([[ 0.4472, -0.8944],
        [ 0.8944,  0.4472]])


In [134]:
# explained variance ratio:

$$\lambda_1 = 8, \quad \lambda_2 = 2$$

$$8 + 2 = 10$$

$$\text{EVR}_i = \frac{\lambda_i}{\sum_{j} \lambda_j}$$

$$\text{EVR}_1 = \frac{8}{10} = 0.8$$

$$80\%$$

$$\text{EVR}_2 = \frac{2}{10} = 0.2$$

$$20\%$$

In [135]:
# tell how much of original features does it hold?
# PC1 → 40%
# PC2 → 25%
# PC3 → 15%
# PC4 → 8%
# PC5 → 4%
# ...

# supports cummulative
# PC1              40%
# PC1 + PC2        65%
# PC1 + PC2 + PC3  80%
# PC1 ... PC4      88%
# PC1 ... PC5      92%

In [136]:
evr = (eigenval / eigenval.sum())
evr

tensor([1., 0.])

In [137]:
pc1 = eigenvec[:, 0]
pc1

tensor([0.4472, 0.8944])

In [138]:
X_red = X_c @ pc1
X_red

tensor([-4.4721, -2.2361,  0.0000,  2.2361,  4.4721])

In [139]:
cumulative_variance = torch.cumsum(evr, dim=0)
cumulative_variance

tensor([1., 1.])

In [140]:
# until now lets revise.
# first take the data, then X = (X-mean) it
# then find its covariance, here by using (X^T x X) // N
# this is how covariance matrix looks like

$$\boxed{\Sigma = \begin{bmatrix} \text{Var}(X) & \text{Cov}(X, Y) \\ \text{Cov}(X, Y) & \text{Var}(Y) \end{bmatrix}}$$

In [141]:
# now we bring out all the eigen stuff from it,
# we use torch.linalg.eigh(cov_matrix), bam we get those two values, now we have to just sort eigenvalues in desc
# we use .flip(dim) since via the above linalg.eigh thing the data produced comes sorted already, and we just had to flip it one way so its desc
# hand written way, Σw = λw, and Σ would be the cov_matrix. and so to bring out λ we shall do det(Σ-λI) = 0
# then we have to just solve for λ. the values we get for λ will be the eigenvalues

In [142]:
class PCA:
    def __init__(self, n_comp):
        self.n_comp = n_comp

    def fit(self, X):
        self.mean = X.mean()
        X_c = X - self.mean

        cov_matrix = (X_c.T @ X_c) / len(X)

        eigval, eigvec = torch.linalg.eigh(cov_matrix)

        # sorting
        eigval = eigval.flip(0)
        eigvec = eigvec.flip(1)

        self.eigval = eigval[:self.n_comp]
        self.comp = eigvec[:, :self.n_comp]
        
        self.explained_variance_ratio = (self.eigval / eigval.sum())

        return self

    def transform(self, X):
        X_c = X-self.mean
        return X_c @ self.comp

In [143]:
pca = PCA(n_comp=1)
pca.fit(X)
X_reduced = pca.transform(X)
print(X_reduced)

tensor([[-3.3268],
        [-1.1351],
        [ 1.0565],
        [ 3.2482],
        [ 5.4399]])
